# 01 · Centralized baseline  (Phase **P1**)

Trains the **centralized upper-bound** model: one ImageNet-pretrained **ResNet-18** on the *union*
of all clients' training data, using the **exact partition saved in notebook 00**. This is the
number every federated regime will be compared against (`PLAN.md` §6, `TODO.md` P1).

**What P1 delivers / fixes:**
- **Gate G3 — overfit-one-batch:** proves the model+loss+data pipeline can actually learn.
- Trains with **SGD + cosine schedule** (the stable configuration; the old project's Adam +
  many local epochs is what made FL collapse — see B5).
- Reports the **full metric suite on the held-out TEST set from the same model** — accuracy,
  macro-F1, per-class F1, confusion matrix, Cohen's κ (kills the val-vs-test and throwaway-probe
  bugs B7/B10).
- Saves config + metrics + checkpoint + data/git hash to one results folder (kills B19).

**Expected result:** EuroSAT centralized accuracy in the **~95%+** ballpark (literature yardstick).

> Run `00_setup_and_eda.ipynb` first (it creates the partition this notebook loads).


## 1 · Environment + Drive (same as notebook 00)

In [ ]:
import os, sys, subprocess

# --- Get the fedsat package (clone the repo) ---
REPO_URL = "https://github.com/sgogoigh/Satellite-Image-Classification-Fed-Learning.git"
REPO_DIR = "/content/Satellite-Image-Classification-Fed-Learning"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)

# --- Install deps (torch/torchvision already ship with Colab; do NOT reinstall them) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# --- Make the package importable ---
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import torch, fedsat
from fedsat.utils import get_device
print("fedsat", fedsat.__version__, "| torch", torch.__version__, "| device:", get_device())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run this cell.")


In [ ]:
# Optional but recommended: persist the HF cache, partitions, and results on Drive
# so you don't re-download / re-partition every session.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/fedsat"
except Exception as e:
    print("Drive not mounted (", e, ") -> falling back to local ./artifacts")
    DRIVE = os.path.join(REPO_DIR, "artifacts")
os.makedirs(DRIVE, exist_ok=True)
print("Artifacts dir:", DRIVE)


## 2 · Configuration — must match notebook 00's partition keys

`dataset / num_clients / alpha / seed` must equal what you used in P0 so the saved partition loads.
Only `experiment_name` and training fields differ here.


In [ ]:
from fedsat.config import ExperimentConfig
from fedsat.utils import set_seed, get_device

cfg = ExperimentConfig(
    experiment_name="p1_centralized",
    dataset="eurosat_rgb", hf_repo="blanchon/EuroSAT_RGB",
    num_clients=10, alpha=0.5, seed=42,     # <-- must match notebook 00
    input_size=64,
    backbone="resnet18", pretrained=True, norm="bn",
    optimizer="sgd", lr=0.01, momentum=0.9, weight_decay=5e-4,
    batch_size=64, max_epochs=40, early_stop_patience=7, lr_schedule="cosine",
    data_cache_dir=f"{DRIVE}/hf_cache",
    partition_dir=f"{DRIVE}/partitions",
    results_dir=f"{DRIVE}/results",
    device=get_device(),
)
set_seed(cfg.seed)
device = cfg.device
print(cfg)


## 3 · Load data + the saved partition

In [ ]:
from fedsat.data import load_eurosat, load_partition

hf_ds, class_names, labels = load_eurosat(cfg.hf_repo, cache_dir=cfg.data_cache_dir)
try:
    partition = load_partition(cfg)
except FileNotFoundError:
    raise FileNotFoundError(
        "Partition not found. Run 00_setup_and_eda.ipynb first with the SAME "
        f"dataset/num_clients/alpha/seed. Expected: {cfg.partition_path()}")

print("loaded partition:", cfg.partition_path())
print("data_hash:", partition["meta"]["data_hash"], "| global_test:", len(partition["global_test"]))


## 4 · **Gate G3** — overfit a single batch

A correct model/loss/data pipeline drives one batch to ~100% accuracy in a few dozen steps.
If this fails, stop and fix the pipeline before training for real.


In [ ]:
from fedsat.models import build_model
from fedsat.engine import overfit_one_batch
from fedsat.data import make_loader, pooled_indices

train_idx = pooled_indices(partition, "train")   # union of all clients' train splits
probe_loader = make_loader(hf_ds, train_idx[:512], cfg, train=True)
probe = build_model(cfg.backbone, cfg.num_classes, cfg.pretrained, cfg.in_channels, cfg.norm).to(device)

acc = overfit_one_batch(probe, probe_loader, device, steps=60)
print(f"G3 overfit-one-batch accuracy = {acc:.3f}")
assert acc > 0.90, "G3 FAIL: model cannot fit a single batch -- inspect model/loss/data."
print("PASSED G3")
del probe


## 5 · Train the centralized model

In [ ]:
from fedsat.engine import fit

val_idx = pooled_indices(partition, "val")
train_loader = make_loader(hf_ds, train_idx, cfg, train=True)
val_loader = make_loader(hf_ds, val_idx, cfg, train=False)

model = build_model(cfg.backbone, cfg.num_classes, cfg.pretrained, cfg.in_channels, cfg.norm).to(device)
print(f"training on {len(train_idx)} images | validating on {len(val_idx)} ...")
history = fit(model, train_loader, val_loader, cfg, device)
print("\nbest_val_acc:", round(history["best_val_acc"], 4),
      "| epochs_run:", history["epochs_run"], "| grad_steps:", history["grad_steps"])


## 6 · Evaluate on the **global test set** (comparable across regimes)

The same trained model is scored once on the held-out global test set. These are the numbers the
federated regimes must be compared against.


In [ ]:
from fedsat.engine import evaluate

gtest_loader = make_loader(hf_ds, partition["global_test"], cfg, train=False)
metrics = evaluate(model, gtest_loader, device, cfg.num_classes, class_names)

print("=== Centralized -- global test metrics ===")
print("accuracy        :", round(metrics["accuracy"], 4))
print("balanced accuracy:", round(metrics["balanced_accuracy"], 4))
print("macro F1        :", round(metrics["macro_f1"], 4))
print("cohen kappa     :", round(metrics["cohen_kappa"], 4))


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns

cm = np.array(metrics["confusion_matrix"])
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Centralized -- confusion (global test)")
plt.tight_layout(); plt.show()

pcf = metrics["per_class_f1"]
plt.figure(figsize=(10, 4))
plt.bar(list(pcf.keys()), list(pcf.values()), color="#229954")
plt.ylim(0, 1); plt.ylabel("F1"); plt.xticks(rotation=45, ha="right")
plt.title("Per-class F1 (global test)"); plt.tight_layout(); plt.show()


## 7 · Persist results (one canonical folder per run)

In [ ]:
import torch
from fedsat.utils import save_json, git_commit, utc_now

run = cfg.run_dir()
cfg.to_yaml(os.path.join(run, "config.yaml"))
save_json(os.path.join(run, "metrics.json"), {
    "regime": "centralized",
    "metrics": metrics,
    "history": history,
    "data_hash": partition["meta"]["data_hash"],
    "git_commit": git_commit(),
    "created_at": utc_now(),
})
torch.save({"state_dict": model.state_dict(), "config": cfg.__dict__},
           os.path.join(run, "centralized_resnet18.pt"))
print("saved results ->", run)


## Done — P1 complete ✅

If centralized test accuracy is in the **~95%+** range, the data + model + evaluation pipeline is
sound. This is the **upper bound** for the federated experiments.

**Next (Phase P2, the make-or-break gate):** `02_federated_fedavg.ipynb` — implement FedAvg in
**Flower** on the IID partition (α=100) and verify **Gate G4: FedAvg ≈ centralized on IID**. Only
after G4 is green do we move to the non-IID sweep and the proposed method. *(P2 notebook is coming
in the next build step.)*
